# 04 — Validate Silver

## Objective

Validate that the Silver tables were correctly created, cleaned and standardized after the Silver transformation process.

## This notebook performs

- Validation of Silver table existence.
- Validation of row counts.
- Validation of required columns.
- Validation of primary identifiers and logical keys.
- Validation of referential integrity between Silver tables.
- Validation of business rules such as:
  - no negative hours;
  - no invalid costs;
  - no invalid target rates;
  - no null mandatory identifiers;
  - no duplicated logical keys.
- Creation of a persistent Silver quality log table.

## Notes

This notebook does not transform data. It audits the output of `03_transform_silver` and stores validation results as evidence of data quality.

In [0]:
# Import required libraries for validation logging.

from datetime import datetime
from pyspark.sql import functions as F

In [0]:
# Use the project schema where Silver tables were created.

spark.sql("USE SCHEMA workbridge")

DataFrame[]

In [0]:
# Read Silver tables created in the previous notebook.

silver_employees_df = spark.table("silver_employees")
silver_clients_df = spark.table("silver_clients")
silver_assignments_df = spark.table("silver_assignments")
silver_hours_df = spark.table("silver_hours")
silver_costs_df = spark.table("silver_costs")
silver_goals_df = spark.table("silver_goals")
silver_rejected_records_df = spark.table("silver_rejected_records")

In [0]:
# This list will store the result of each Silver validation check.

silver_validation_results = []

In [0]:
# Helper function to register validation results in a consistent structure.

def add_silver_validation_result(table_name, check_name, status, records_count=None, message=""):
    """
    Add a Silver validation result to the silver_validation_results list.

    Parameters:
    - table_name: name of the table being validated.
    - check_name: name of the validation check.
    - status: PASS or FAIL.
    - records_count: number of records involved in the check.
    - message: short explanation of the result.
    """

    silver_validation_results.append({
        "validation_timestamp": datetime.now(),
        "table_name": table_name,
        "check_name": check_name,
        "status": status,
        "records_count": records_count,
        "message": message,
    })

In [0]:
# Validate that all Silver tables exist and contain records.

silver_tables = [
    "silver_employees",
    "silver_clients",
    "silver_assignments",
    "silver_hours",
    "silver_costs",
    "silver_goals",
    "silver_rejected_records",
]

for table_name in silver_tables:
    try:
        df = spark.table(table_name)
        row_count = df.count()

        add_silver_validation_result(
            table_name=table_name,
            check_name="table_exists_check",
            status="PASS",
            records_count=row_count,
            message="Table exists."
        )

        if row_count > 0:
            add_silver_validation_result(
                table_name=table_name,
                check_name="row_count_check",
                status="PASS",
                records_count=row_count,
                message="Table contains records."
            )
        else:
            add_silver_validation_result(
                table_name=table_name,
                check_name="row_count_check",
                status="FAIL",
                records_count=row_count,
                message="Table is empty."
            )

    except Exception as error:
        add_silver_validation_result(
            table_name=table_name,
            check_name="table_exists_check",
            status="FAIL",
            records_count=None,
            message=f"Table could not be read: {str(error)}"
        )

In [0]:
# Validate mandatory identifiers and uniqueness rules in Silver master/detail tables.

# silver_employees: employee_id must not be null and must be unique.
employee_id_null_count = silver_employees_df.filter(
    F.col("employee_id").isNull() | (F.col("employee_id") == "")
).count()

add_silver_validation_result(
    table_name="silver_employees",
    check_name="employee_id_not_null_check",
    status="PASS" if employee_id_null_count == 0 else "FAIL",
    records_count=employee_id_null_count,
    message="employee_id is not null or empty." if employee_id_null_count == 0 else "employee_id contains null or empty values."
)

employee_duplicate_count = (
    silver_employees_df
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_silver_validation_result(
    table_name="silver_employees",
    check_name="employee_id_unique_check",
    status="PASS" if employee_duplicate_count == 0 else "FAIL",
    records_count=employee_duplicate_count,
    message="employee_id is unique." if employee_duplicate_count == 0 else "Duplicated employee_id values found."
)

# silver_clients: client_id must not be null and must be unique.
client_id_null_count = silver_clients_df.filter(
    F.col("client_id").isNull() | (F.col("client_id") == "")
).count()

add_silver_validation_result(
    table_name="silver_clients",
    check_name="client_id_not_null_check",
    status="PASS" if client_id_null_count == 0 else "FAIL",
    records_count=client_id_null_count,
    message="client_id is not null or empty." if client_id_null_count == 0 else "client_id contains null or empty values."
)

client_duplicate_count = (
    silver_clients_df
    .groupBy("client_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_silver_validation_result(
    table_name="silver_clients",
    check_name="client_id_unique_check",
    status="PASS" if client_duplicate_count == 0 else "FAIL",
    records_count=client_duplicate_count,
    message="client_id is unique." if client_duplicate_count == 0 else "Duplicated client_id values found."
)

# silver_assignments: assignment_id must not be null and must be unique.
assignment_id_null_count = silver_assignments_df.filter(
    F.col("assignment_id").isNull() | (F.col("assignment_id") == "")
).count()

add_silver_validation_result(
    table_name="silver_assignments",
    check_name="assignment_id_not_null_check",
    status="PASS" if assignment_id_null_count == 0 else "FAIL",
    records_count=assignment_id_null_count,
    message="assignment_id is not null or empty." if assignment_id_null_count == 0 else "assignment_id contains null or empty values."
)

assignment_duplicate_count = (
    silver_assignments_df
    .groupBy("assignment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_silver_validation_result(
    table_name="silver_assignments",
    check_name="assignment_id_unique_check",
    status="PASS" if assignment_duplicate_count == 0 else "FAIL",
    records_count=assignment_duplicate_count,
    message="assignment_id is unique." if assignment_duplicate_count == 0 else "Duplicated assignment_id values found."
)

In [0]:
# Validate uniqueness of logical composite keys.

# silver_hours logical key: period + employee_id + client_id + project_id
# silver_hours logical key: hours_record_id
# hours_record_id is built as period + employee_id + client_id + project_id.

hours_record_id_null_count = silver_hours_df.filter(
    F.col("hours_record_id").isNull() | (F.col("hours_record_id") == "")
).count()

add_silver_validation_result(
    table_name="silver_hours",
    check_name="hours_record_id_not_null_check",
    status="PASS" if hours_record_id_null_count == 0 else "FAIL",
    records_count=hours_record_id_null_count,
    message="hours_record_id is not null or empty."
    if hours_record_id_null_count == 0 else "hours_record_id contains null or empty values."
)

hours_duplicate_count = (
    silver_hours_df
    .groupBy("hours_record_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_silver_validation_result(
    table_name="silver_hours",
    check_name="hours_record_id_unique_check",
    status="PASS" if hours_duplicate_count == 0 else "FAIL",
    records_count=hours_duplicate_count,
    message="hours_record_id is unique."
    if hours_duplicate_count == 0 else "Duplicated hours_record_id values found."
)


# silver_costs logical key: cost_record_id
cost_duplicate_count = (
    silver_costs_df
    .groupBy("cost_record_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_silver_validation_result(
    table_name="silver_costs",
    check_name="cost_record_id_unique_check",
    status="PASS" if cost_duplicate_count == 0 else "FAIL",
    records_count=cost_duplicate_count,
    message="cost_record_id is unique." if cost_duplicate_count == 0 else "Duplicated cost_record_id values found."
)

# silver_goals logical key: goal_record_id
goal_duplicate_count = (
    silver_goals_df
    .groupBy("goal_record_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_silver_validation_result(
    table_name="silver_goals",
    check_name="goal_record_id_unique_check",
    status="PASS" if goal_duplicate_count == 0 else "FAIL",
    records_count=goal_duplicate_count,
    message="goal_record_id is unique." if goal_duplicate_count == 0 else "Duplicated goal_record_id values found."
)

In [0]:
# Validate referential integrity between Silver tables.

valid_employee_ids_df = silver_employees_df.select("employee_id").distinct()
valid_client_ids_df = silver_clients_df.select("client_id").distinct()

# Assignments → Employees
invalid_assignment_employee_fk_count = (
    silver_assignments_df
    .join(valid_employee_ids_df, on="employee_id", how="left_anti")
    .count()
)

add_silver_validation_result(
    table_name="silver_assignments",
    check_name="employee_fk_check",
    status="PASS" if invalid_assignment_employee_fk_count == 0 else "FAIL",
    records_count=invalid_assignment_employee_fk_count,
    message="All assignment employee_id values exist in silver_employees."
    if invalid_assignment_employee_fk_count == 0 else "Some assignment employee_id values do not exist in silver_employees."
)

# Assignments → Clients
invalid_assignment_client_fk_count = (
    silver_assignments_df
    .join(valid_client_ids_df, on="client_id", how="left_anti")
    .count()
)

add_silver_validation_result(
    table_name="silver_assignments",
    check_name="client_fk_check",
    status="PASS" if invalid_assignment_client_fk_count == 0 else "FAIL",
    records_count=invalid_assignment_client_fk_count,
    message="All assignment client_id values exist in silver_clients."
    if invalid_assignment_client_fk_count == 0 else "Some assignment client_id values do not exist in silver_clients."
)

# Hours → Employees
invalid_hours_employee_fk_count = (
    silver_hours_df
    .join(valid_employee_ids_df, on="employee_id", how="left_anti")
    .count()
)

add_silver_validation_result(
    table_name="silver_hours",
    check_name="employee_fk_check",
    status="PASS" if invalid_hours_employee_fk_count == 0 else "FAIL",
    records_count=invalid_hours_employee_fk_count,
    message="All hours employee_id values exist in silver_employees."
    if invalid_hours_employee_fk_count == 0 else "Some hours employee_id values do not exist in silver_employees."
)

# Hours → Clients
invalid_hours_client_fk_count = (
    silver_hours_df
    .join(valid_client_ids_df, on="client_id", how="left_anti")
    .count()
)

add_silver_validation_result(
    table_name="silver_hours",
    check_name="client_fk_check",
    status="PASS" if invalid_hours_client_fk_count == 0 else "FAIL",
    records_count=invalid_hours_client_fk_count,
    message="All hours client_id values exist in silver_clients."
    if invalid_hours_client_fk_count == 0 else "Some hours client_id values do not exist in silver_clients."
)

# Costs → Employees
invalid_costs_employee_fk_count = (
    silver_costs_df
    .join(valid_employee_ids_df, on="employee_id", how="left_anti")
    .count()
)

add_silver_validation_result(
    table_name="silver_costs",
    check_name="employee_fk_check",
    status="PASS" if invalid_costs_employee_fk_count == 0 else "FAIL",
    records_count=invalid_costs_employee_fk_count,
    message="All costs employee_id values exist in silver_employees."
    if invalid_costs_employee_fk_count == 0 else "Some costs employee_id values do not exist in silver_employees."
)

# Goals → Clients
invalid_goals_client_fk_count = (
    silver_goals_df
    .join(valid_client_ids_df, on="client_id", how="left_anti")
    .count()
)

add_silver_validation_result(
    table_name="silver_goals",
    check_name="client_fk_check",
    status="PASS" if invalid_goals_client_fk_count == 0 else "FAIL",
    records_count=invalid_goals_client_fk_count,
    message="All goals client_id values exist in silver_clients."
    if invalid_goals_client_fk_count == 0 else "Some goals client_id values do not exist in silver_clients."
)

In [0]:
# Validate business rules in Silver tables.

# Employees: region_id is monitored as a warning-level data quality indicator.
# Employees with null region_id are not rejected in Silver because the employee
# can still be used in other analyses, and regional reporting can also rely on
# client or cost region depending on the metric.

employee_region_null_count = silver_employees_df.filter(F.col("region_id").isNull()).count()

add_silver_validation_result(
    table_name="silver_employees",
    check_name="employee_region_null_warning_check",
    status="PASS",
    records_count=employee_region_null_count,
    message=f"{employee_region_null_count} employees have null region_id and should be reviewed."
)
invalid_employee_status_count = silver_employees_df.filter(F.col("employment_status").isNull()).count()

add_silver_validation_result(
    table_name="silver_employees",
    check_name="employment_status_not_null_check",
    status="PASS" if invalid_employee_status_count == 0 else "FAIL",
    records_count=invalid_employee_status_count,
    message="All employees have a valid employment_status." if invalid_employee_status_count == 0 else "Some employees have null employment_status."
)

# Clients: region_id and contract_type should not be null.
invalid_client_region_count = silver_clients_df.filter(F.col("region_id").isNull()).count()

add_silver_validation_result(
    table_name="silver_clients",
    check_name="client_region_not_null_check",
    status="PASS" if invalid_client_region_count == 0 else "FAIL",
    records_count=invalid_client_region_count,
    message="All clients have a valid region_id." if invalid_client_region_count == 0 else "Some clients have null region_id."
)

invalid_client_contract_count = silver_clients_df.filter(F.col("contract_type").isNull()).count()

add_silver_validation_result(
    table_name="silver_clients",
    check_name="contract_type_not_null_check",
    status="PASS" if invalid_client_contract_count == 0 else "FAIL",
    records_count=invalid_client_contract_count,
    message="All clients have a valid contract_type." if invalid_client_contract_count == 0 else "Some clients have null contract_type."
)

# Assignments: allocation must be between 0 and 100.
invalid_assignment_allocation_count = silver_assignments_df.filter(
    (F.col("allocation_percentage").isNull())
    | (F.col("allocation_percentage") <= 0)
    | (F.col("allocation_percentage") > 100)
).count()

add_silver_validation_result(
    table_name="silver_assignments",
    check_name="allocation_percentage_range_check",
    status="PASS" if invalid_assignment_allocation_count == 0 else "FAIL",
    records_count=invalid_assignment_allocation_count,
    message="All assignment allocation percentages are valid."
    if invalid_assignment_allocation_count == 0 else "Invalid allocation percentages found."
)

# Hours: no negative values and billable_hours <= worked_hours.
invalid_hours_count = silver_hours_df.filter(
    (F.col("available_hours").isNull()) | (F.col("available_hours") <= 0)
    | (F.col("worked_hours").isNull()) | (F.col("worked_hours") < 0)
    | (F.col("billable_hours").isNull()) | (F.col("billable_hours") < 0)
    | (F.col("billable_hours") > F.col("worked_hours"))
    | (F.col("non_billable_hours").isNull()) | (F.col("non_billable_hours") < 0)
    | (F.col("overtime_hours").isNull()) | (F.col("overtime_hours") < 0)
).count()

add_silver_validation_result(
    table_name="silver_hours",
    check_name="hours_metrics_validity_check",
    status="PASS" if invalid_hours_count == 0 else "FAIL",
    records_count=invalid_hours_count,
    message="All hour metrics are valid." if invalid_hours_count == 0 else "Invalid hour metric values found."
)

# Costs: no negative costs and total_cost must match salary_cost + benefits_cost.
invalid_costs_count = silver_costs_df.filter(
    (F.col("salary_cost").isNull()) | (F.col("salary_cost") < 0)
    | (F.col("benefits_cost").isNull()) | (F.col("benefits_cost") < 0)
    | (F.col("total_cost").isNull()) | (F.col("total_cost") < 0)
    | (F.abs(F.col("total_cost") - (F.col("salary_cost") + F.col("benefits_cost"))) > 0.01)
).count()

add_silver_validation_result(
    table_name="silver_costs",
    check_name="cost_metrics_validity_check",
    status="PASS" if invalid_costs_count == 0 else "FAIL",
    records_count=invalid_costs_count,
    message="All cost metrics are valid." if invalid_costs_count == 0 else "Invalid cost metric values found."
)

# Goals: target rates between 0 and 1, and target values non-negative.
invalid_goals_count = silver_goals_df.filter(
    (F.col("target_turnover_rate").isNull()) | (F.col("target_turnover_rate") < 0) | (F.col("target_turnover_rate") > 1)
    | (F.col("target_utilization_rate").isNull()) | (F.col("target_utilization_rate") < 0) | (F.col("target_utilization_rate") > 1)
    | (F.col("target_cost").isNull()) | (F.col("target_cost") < 0)
    | (F.col("target_billable_hours").isNull()) | (F.col("target_billable_hours") < 0)
).count()

add_silver_validation_result(
    table_name="silver_goals",
    check_name="goal_metrics_validity_check",
    status="PASS" if invalid_goals_count == 0 else "FAIL",
    records_count=invalid_goals_count,
    message="All goal metrics are valid." if invalid_goals_count == 0 else "Invalid goal metric values found."
)

In [0]:
# Convert validation results into a Spark DataFrame.

silver_quality_log_df = spark.createDataFrame(silver_validation_results)

display(silver_quality_log_df)

check_name,message,records_count,status,table_name,validation_timestamp
table_exists_check,Table exists.,493,PASS,silver_employees,2026-05-09T12:59:03.617Z
row_count_check,Table contains records.,493,PASS,silver_employees,2026-05-09T12:59:03.617Z
table_exists_check,Table exists.,13,PASS,silver_clients,2026-05-09T12:59:04.361Z
row_count_check,Table contains records.,13,PASS,silver_clients,2026-05-09T12:59:04.361Z
table_exists_check,Table exists.,435,PASS,silver_assignments,2026-05-09T12:59:04.720Z
row_count_check,Table contains records.,435,PASS,silver_assignments,2026-05-09T12:59:04.720Z
table_exists_check,Table exists.,3292,PASS,silver_hours,2026-05-09T12:59:05.111Z
row_count_check,Table contains records.,3292,PASS,silver_hours,2026-05-09T12:59:05.111Z
table_exists_check,Table exists.,4865,PASS,silver_costs,2026-05-09T12:59:05.464Z
row_count_check,Table contains records.,4865,PASS,silver_costs,2026-05-09T12:59:05.464Z


In [0]:
# Save Silver validation results as a Delta table.

silver_quality_log_df.write.mode("overwrite").format("delta").saveAsTable("silver_quality_log")

In [0]:
# Show validation summary by status.

silver_validation_summary_df = (
    silver_quality_log_df
    .groupBy("status")
    .agg(F.count("*").alias("checks_count"))
)

display(silver_validation_summary_df)

status,checks_count
PASS,38
